# DecisionTree - Model Experiments

This notebook runs all DecisionTree experiments and logs them to the MLflow
experiment **`DecisionTree_Training`** on DagsHub.

## Notebook structure (required by the assignment)

1. Setup & MLflow Connection
2. Cleaning
3. Feature Engineering
4. Feature Selection
5. Training (incl. hyperparameter sweep + over-/under-fitting demos)
6. Final Pipeline & Logging


# 1. Setup & MLflow Connection


In [ ]:
# =============================================================
# Setup - auto-detects environment and locates the `src/` package.
# See model_experiment_LogisticRegression.ipynb for the full notes.
# =============================================================
import os, sys, subprocess, shutil

REPO_URL = "https://github.com/ekatsirekidze/ML_HW2.git"   # only used for git-clone fallback

ON_KAGGLE = os.path.exists("/kaggle/input")
SRC_FOUND = None

if ON_KAGGLE:
    for _d in os.listdir("/kaggle/input"):
        _candidate = f"/kaggle/input/{_d}"
        if os.path.isdir(os.path.join(_candidate, "src")):
            SRC_FOUND = _candidate
            break
    if SRC_FOUND is None:
        REPO_DIR = "/kaggle/working/repo"
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        subprocess.check_call(["git", "clone", "-q", REPO_URL, REPO_DIR])
        SRC_FOUND = REPO_DIR
    sys.path.insert(0, SRC_FOUND)

    for _pkg in ("mlflow==2.10.2", "dagshub"):
        try:
            __import__(_pkg.split("==")[0].replace("-", "_"))
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _k in ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
        os.environ[_k] = _s.get_secret(_k)

    print("src/ found at :", SRC_FOUND)
else:
    sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit

from src.data import load_train, downcast_numerics
from src.preprocessing import IEEECleaner
from src.feature_engineering import IEEEFeatureEngineer
from src.feature_selection import (
    find_high_correlation, compute_mutual_info, ColumnSubsetSelector,
)
from src.mlflow_utils import (
    setup_mlflow, log_run, log_metrics_dict,
    evaluate_holdout, cross_validate_auc,
    plot_roc, plot_confusion, plot_feature_importance,
)
import mlflow, mlflow.sklearn

setup_mlflow("DecisionTree_Training")
print("Environment :", "Kaggle" if ON_KAGGLE else "Local")
print("Tracking URI:", mlflow.get_tracking_uri())


## 1.1 Load data and create a time-based holdout

Same setup as the LR notebook: sort by `TransactionDT`, hold out the
last 20% in time.  Most exploratory runs use a 100k stratified subsample;
the *final* model retrains on full data.


In [ ]:
RANDOM_STATE  = 42
SUBSAMPLE     = 100_000
HOLDOUT_FRAC  = 0.20

df = downcast_numerics(load_train())
df = df.sort_values("TransactionDT").reset_index(drop=True)

split_idx = int(len(df) * (1 - HOLDOUT_FRAC))
train_full = df.iloc[:split_idx]
val_full   = df.iloc[split_idx:]

def subsample_stratified(d, n, seed=RANDOM_STATE):
    if len(d) <= n:
        return d
    pos = d[d["isFraud"] == 1]
    neg_pool = d[d["isFraud"] == 0]
    need = n - len(pos)
    neg = neg_pool.sample(need, random_state=seed) if need < len(neg_pool) else neg_pool
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)

train_small = subsample_stratified(train_full, SUBSAMPLE)

X_train_s, y_train_s = train_small.drop(columns=["isFraud"]), train_small["isFraud"]
X_val,     y_val     = val_full.drop(columns=["isFraud"]),    val_full["isFraud"]
X_train_f, y_train_f = train_full.drop(columns=["isFraud"]),  train_full["isFraud"]

print(f"train (full)  : {X_train_f.shape}  | fraud rate {y_train_f.mean():.3%}")
print(f"train (small) : {X_train_s.shape}  | fraud rate {y_train_s.mean():.3%}")
print(f"val           : {X_val.shape}  | fraud rate {y_val.mean():.3%}")


## 1.2 Pipeline factory

Same factory pattern as the LR notebook.  Differences for DecisionTree:

- numeric: constant impute (-999), no scaling
- categorical: ordinal-encoded (one-hot would slow trees down)


In [ ]:
def make_decisiontree_pipeline(
    *,
    cleaner_kwargs: dict | None = None,
    fe_kwargs:      dict | None = None,
    model_kwargs:   dict | None = None,
) -> Pipeline:
    cleaner_kwargs = cleaner_kwargs or {}
    fe_kwargs      = fe_kwargs      or {}
    base_kwargs    = dict(max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE, class_weight=None)
    base_kwargs.update(model_kwargs or {})

    from sklearn.preprocessing import OrdinalEncoder
    num_pipe = SimpleImputer(strategy="constant", fill_value=-999.0)
    cat_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("ord",    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_pipe, make_column_selector(dtype_include=np.number)),
            ("cat", cat_pipe, make_column_selector(dtype_include=object)),
        ],
        remainder="drop", verbose_feature_names_out=False,
    )

    return Pipeline([
        ("clean", IEEECleaner(**cleaner_kwargs)),
        ("fe",    IEEEFeatureEngineer(**fe_kwargs)),
        ("prep",  preprocessor),
        ("model", DecisionTreeClassifier(**base_kwargs)),
    ])

# Smoke-test
print(make_decisiontree_pipeline())


# 2. Cleaning

Three cleaning strategies (same as LR notebook so results compare cleanly).


In [ ]:
cleaning_grid = [
    dict(run="cleaning_v1_drop95",  cleaner=dict(missing_threshold=0.95)),
    dict(run="cleaning_v2_drop90",  cleaner=dict(missing_threshold=0.90)),
    dict(run="cleaning_v3_drop99",  cleaner=dict(missing_threshold=0.99)),
]

cleaning_results = {}
for cfg_ in cleaning_grid:
    with log_run(
        cfg_["run"],
        params={**cfg_["cleaner"], "subsample": SUBSAMPLE, "fe_active": False},
        tags={"model_family": "DecisionTree", "stage": "cleaning"},
    ):
        pipe = make_decisiontree_pipeline(
            cleaner_kwargs=cfg_["cleaner"],
            fe_kwargs=dict(use_datetime=False, use_email=False,
                            use_freq=False, use_agg=False),
        )
        m = evaluate_holdout(pipe, X_train_s, y_train_s, X_val, y_val)
        log_metrics_dict(m)
        cleaning_results[cfg_["run"]] = m
        print(f"{cfg_['run']:25} train={m['train_auc']:.4f}  val={m['val_auc']:.4f}  gap={m['gap']:+.4f}")

best_cleaning      = max(cleaning_results, key=lambda k: cleaning_results[k]["val_auc"])
BEST_CLEANING_KW   = next(c for c in cleaning_grid if c["run"] == best_cleaning)["cleaner"]
print(f"\n>> best cleaning: {best_cleaning}")


# 3. Feature Engineering

Cumulative FE blocks (same five rungs as in the LR notebook).


In [ ]:
fe_grid = [
    ("fe_v1_baseline",  dict(use_datetime=False, use_email=False, use_freq=False, use_agg=False)),
    ("fe_v2_+datetime", dict(use_datetime=True,  use_email=False, use_freq=False, use_agg=False)),
    ("fe_v3_+email",    dict(use_datetime=True,  use_email=True,  use_freq=False, use_agg=False)),
    ("fe_v4_+freq",     dict(use_datetime=True,  use_email=True,  use_freq=True,  use_agg=False)),
    ("fe_v5_full",      dict(use_datetime=True,  use_email=True,  use_freq=True,  use_agg=True)),
]

fe_results = {}
for run_name, fe_kw in fe_grid:
    with log_run(
        run_name,
        params={**fe_kw, "subsample": SUBSAMPLE, "cleaning": best_cleaning},
        tags={"model_family": "DecisionTree", "stage": "fe"},
    ):
        pipe = make_decisiontree_pipeline(
            cleaner_kwargs=BEST_CLEANING_KW, fe_kwargs=fe_kw,
        )
        m = evaluate_holdout(pipe, X_train_s, y_train_s, X_val, y_val)
        log_metrics_dict(m)
        fe_results[run_name] = m
        print(f"{run_name:20} train={m['train_auc']:.4f}  val={m['val_auc']:.4f}  gap={m['gap']:+.4f}")

best_fe_name = max(fe_results, key=lambda k: fe_results[k]["val_auc"])
BEST_FE_KW   = dict(fe_grid)[best_fe_name]
print(f"\n>> best FE: {best_fe_name}")


# 4. Feature Selection

Four strategies, each operating on the source-column level.


In [ ]:
# Materialise the cleaned + FE'd training subsample for selection analysis.
prep_view = Pipeline([
    ("clean", IEEECleaner(**BEST_CLEANING_KW)),
    ("fe",    IEEEFeatureEngineer(**BEST_FE_KW)),
]).fit(X_train_s, y_train_s).transform(X_train_s)

cat_cols_src = prep_view.select_dtypes(include="object").columns.tolist()
num_cols_src = prep_view.select_dtypes(include=np.number).columns.tolist()

all_cols = prep_view.columns.tolist()

high_corr = find_high_correlation(prep_view[num_cols_src], threshold=0.95)
fs_v2_keep = [c for c in all_cols if c not in high_corr]

mi = compute_mutual_info(prep_view[num_cols_src], y_train_s, sample=50_000)
fs_v3_keep = list(dict.fromkeys(mi.head(200).index.tolist() + cat_cols_src))

# Tree-importance based selection
from sklearn.ensemble import RandomForestClassifier
rf_probe = RandomForestClassifier(
    n_estimators=80, max_depth=12, n_jobs=-1, random_state=RANDOM_STATE,
)
rf_probe.fit(prep_view[num_cols_src].fillna(-999), y_train_s)
rf_imp = pd.Series(rf_probe.feature_importances_, index=num_cols_src)
fs_v4_keep = list(dict.fromkeys(
    rf_imp.sort_values(ascending=False).head(200).index.tolist() + cat_cols_src
))

print(f"v1_all          : {len(all_cols):4d}")
print(f"v2_drop_corr95  : {len(fs_v2_keep):4d}")
print(f"v3_top_mi200    : {len(fs_v3_keep):4d}")
print(f"v4_top_rfimp200 : {len(fs_v4_keep):4d}")


In [ ]:
def make_pipeline_with_subset(keep_cols, **kw):
    base = make_decisiontree_pipeline(**kw)
    if keep_cols is None:
        return base
    steps = list(base.steps)
    fe_idx = next(i for i, (n, _) in enumerate(steps) if n == "fe")
    steps.insert(fe_idx + 1, ("subset", ColumnSubsetSelector(keep_cols)))
    return Pipeline(steps)

fs_grid = [
    ("fs_v1_all",           None),
    ("fs_v2_drop_corr95",   fs_v2_keep),
    ("fs_v3_top_mi200",     fs_v3_keep),
    ("fs_v4_top_rfimp200",  fs_v4_keep),
]

fs_results = {}
for run_name, keep in fs_grid:
    with log_run(
        run_name,
        params={"n_features_kept": (len(keep) if keep else len(all_cols)),
                "subsample": SUBSAMPLE,
                "cleaning":  best_cleaning,
                "fe":        best_fe_name},
        tags={"model_family": "DecisionTree", "stage": "fs"},
    ):
        pipe = make_pipeline_with_subset(
            keep, cleaner_kwargs=BEST_CLEANING_KW, fe_kwargs=BEST_FE_KW,
        )
        m = evaluate_holdout(pipe, X_train_s, y_train_s, X_val, y_val)
        log_metrics_dict(m)
        fs_results[run_name] = m
        print(f"{run_name:22} train={m['train_auc']:.4f}  val={m['val_auc']:.4f}  gap={m['gap']:+.4f}")

best_fs_name = max(fs_results, key=lambda k: fs_results[k]["val_auc"])
BEST_FS_KEEP = dict(fs_grid)[best_fs_name]
print(f"\n>> best FS: {best_fs_name}")


# 5. Training

Now cleaning + FE + FS are locked in.  We sweep DecisionTree hyperparameters
and explicitly include over-/under-fit demos.


In [ ]:
# --- Precompute the heavy cleaning + FE + FS + preprocessing once -------
# Section 5 only varies the DecisionTree hyperparameters; cleaning / FE / FS
# / preprocessing are identical for every run, so re-running them inside
# every pipeline wastes minutes per run.  We build the pipeline once,
# drop the final model step, fit on the training subsample, transform
# both train + holdout to matrices, and then per-experiment fit only
# the DecisionTreeClassifier(...) step on the precomputed matrices.  The MLflow logs
# (params, metrics, plots) are byte-for-byte identical to the
# pipeline-based version.
import time
from sklearn.metrics import roc_auc_score

print("Precomputing transformed matrices for the DecisionTree training sweep ...")
_t0 = time.time()
_prep_only = make_pipeline_with_subset(
    BEST_FS_KEEP,
    cleaner_kwargs=BEST_CLEANING_KW,
    fe_kwargs=BEST_FE_KW,
    model_kwargs=None,
)
_prep_only.steps = _prep_only.steps[:-1]                  # drop the model step
_prep_only.fit(X_train_s, y_train_s)
X_tr_mat  = _prep_only.transform(X_train_s)
X_val_mat = _prep_only.transform(X_val)
print(f"  done in {time.time()-_t0:.1f}s | train shape={X_tr_mat.shape}")

MODEL_DEFAULT_KW = dict(max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE, class_weight=None)


def run_lr_training(run_name, model_kwargs, *, purpose=None, extra_params=None):
    base_kwargs = dict(MODEL_DEFAULT_KW)
    base_kwargs.update(model_kwargs or {})
    model = DecisionTreeClassifier(**base_kwargs)

    with log_run(
        run_name,
        params={
            **{k: str(v) for k, v in model_kwargs.items()},
            "subsample":      SUBSAMPLE,
            "cleaning":       best_cleaning,
            "fe":             best_fe_name,
            "feature_select": best_fs_name,
            **(extra_params or {}),
        },
        tags={"model_family": "DecisionTree", "stage": "training",
                **({"purpose": purpose} if purpose else {})},
    ):
        t0 = time.time()
        model.fit(X_tr_mat, y_train_s)
        fit_sec = time.time() - t0
        p_tr  = model.predict_proba(X_tr_mat)[:, 1]
        p_val = model.predict_proba(X_val_mat)[:, 1]
        tr_auc  = float(roc_auc_score(y_train_s, p_tr))
        val_auc = float(roc_auc_score(y_val,     p_val))
        m = {"train_auc": tr_auc, "val_auc": val_auc,
              "gap": tr_auc - val_auc, "fit_sec": float(fit_sec)}
        log_metrics_dict(m)
        mlflow.log_figure(plot_roc(y_val, p_val, title=run_name), "roc.png")
        mlflow.log_figure(plot_confusion(y_val, (p_val >= 0.5).astype(int),
                                          title=run_name), "confusion.png")
        print(f"{run_name:34} train={tr_auc:.4f}  val={val_auc:.4f}  "
              f"gap={tr_auc-val_auc:+.4f}  ({fit_sec:.1f}s)")
        return m

run_lr_training("train_v1_baseline_depth8", dict(max_depth=8, min_samples_leaf=20))
run_lr_training("train_v2_depth2_underfit", dict(max_depth=2, min_samples_leaf=20), purpose="underfit_demo")
run_lr_training("train_v3_depthNone_overfit", dict(max_depth=None, min_samples_leaf=1), purpose="overfit_demo")
run_lr_training("train_v4_balanced", dict(max_depth=8, min_samples_leaf=20, class_weight="balanced"))
run_lr_training("train_v5_min_leaf_5", dict(max_depth=10, min_samples_leaf=5))
run_lr_training("train_v6_entropy", dict(max_depth=8, min_samples_leaf=20, criterion="entropy"))


In [ ]:
# --- HPO sweep: reuses the X_tr_mat / X_val_mat from the cell above ---
sweep_grid = [3, 5, 8, 12, 20, None]
sweep_rows = []

with log_run(
    "train_v7_max_depth_sweep_parent",
    params={"sweep_param": "max_depth", "sweep_grid": str(sweep_grid)},
    tags={"model_family": "DecisionTree", "stage": "hpo"},
):
    for v in sweep_grid:
        with log_run(
            f"train_v7_max_depth={v}",
            params={"max_depth": v, **{k: str(vv) for k, vv in dict(min_samples_leaf=20).items()}},
            tags={"model_family": "DecisionTree", "stage": "hpo", "parent": "max_depth_sweep"},
            nested=True,
        ):
            base_kwargs = dict(MODEL_DEFAULT_KW)
            base_kwargs.update(dict(max_depth=v, min_samples_leaf=20))
            model = DecisionTreeClassifier(**base_kwargs)
            t0 = time.time()
            model.fit(X_tr_mat, y_train_s)
            fit_sec = time.time() - t0
            p_tr  = model.predict_proba(X_tr_mat)[:, 1]
            p_val = model.predict_proba(X_val_mat)[:, 1]
            tr_auc  = float(roc_auc_score(y_train_s, p_tr))
            val_auc = float(roc_auc_score(y_val,     p_val))
            m = {"train_auc": tr_auc, "val_auc": val_auc,
                  "gap": tr_auc - val_auc, "fit_sec": float(fit_sec)}
            log_metrics_dict(m)
            sweep_rows.append({"max_depth": v, **m})
            print(f"  max_depth={v}  train={tr_auc:.4f}  val={val_auc:.4f}  "
                  f"gap={tr_auc-val_auc:+.4f}  ({fit_sec:.1f}s)")

sweep_df = pd.DataFrame(sweep_rows).sort_values("val_auc", ascending=False)
sweep_df


# 6. Final Pipeline & Logging

Refit the chosen configuration on the *full* training set and register
it as **`DecisionTree_Fraud_Pipeline`** in the MLflow Model Registry.
Cross-validate with both StratifiedKFold and TimeSeriesSplit to expose
any temporal-leakage gap.


In [ ]:
FINAL_KW = dict(max_depth=8, min_samples_leaf=20)
final_pipe = make_pipeline_with_subset(
    BEST_FS_KEEP,
    cleaner_kwargs=BEST_CLEANING_KW,
    fe_kwargs=BEST_FE_KW,
    model_kwargs=FINAL_KW,
)
print("Final config:", FINAL_KW)


In [ ]:
with log_run(
    "cv_v1_stratified_5fold",
    params={"cv": "StratifiedKFold", "n_splits": 5,
             **{k: str(v) for k, v in FINAL_KW.items()}},
    tags={"model_family": "DecisionTree", "stage": "cv"},
):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    m = cross_validate_auc(final_pipe, X_train_s, y_train_s, cv=skf)
    log_metrics_dict(m)
    print("Stratified KFold:  val_auc =", m["val_auc"], "+/-", m["val_auc_std"])

with log_run(
    "cv_v2_timeseries_5fold",
    params={"cv": "TimeSeriesSplit", "n_splits": 5,
             **{k: str(v) for k, v in FINAL_KW.items()}},
    tags={"model_family": "DecisionTree", "stage": "cv"},
):
    tss = TimeSeriesSplit(n_splits=5)
    m = cross_validate_auc(final_pipe, X_train_s, y_train_s, cv=tss)
    log_metrics_dict(m)
    print("TimeSeriesSplit :  val_auc =", m["val_auc"], "+/-", m["val_auc_std"])


In [ ]:
with log_run(
    "final_pipeline_full_train",
    params={
        **{k: str(v) for k, v in FINAL_KW.items()},
        "cleaning":       best_cleaning,
        "fe":             best_fe_name,
        "feature_select": best_fs_name,
        "trained_on":     "full_train",
    },
    tags={"model_family": "DecisionTree", "stage": "final",
            "purpose": "register_in_model_registry"},
):
    m = evaluate_holdout(final_pipe, X_train_f, y_train_f, X_val, y_val)
    log_metrics_dict(m)
    proba = final_pipe.predict_proba(X_val)[:, 1]
    mlflow.log_figure(plot_roc(y_val, proba, title="Final DecisionTree - holdout ROC"),
                      "roc_final.png")
    mlflow.log_figure(plot_confusion(y_val, (proba >= 0.5).astype(int),
                                      title="Final DecisionTree - confusion @ 0.5"),
                      "confusion_final.png")
    mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="pipeline",
        registered_model_name="DecisionTree_Fraud_Pipeline",
    )
    print(f"Registered model.  holdout val_auc = {m['val_auc']:.4f}  gap = {m['gap']:+.4f}")


## What we learned (write this up in the README)

Paste the actual numbers from MLflow:

- **Best cleaning** for DecisionTree: `{best_cleaning}` -- val_auc on subsample
- **Best FE**: `{best_fe_name}`
- **Best FS**: `{best_fs_name}` -- # features kept
- **Underfit demo**: which run, train_auc, val_auc
- **Overfit demo**: which run, train_auc, val_auc, gap
- **CV gap signal**: TimeSeries vs Stratified val_auc difference
- **Final holdout AUC**: full-data refit number, used in the cross-model comparison
